In [5]:
import pyarrow.parquet as pq
import pyarrow
import os
import time

In [2]:
path = 'dataset'
dest  = 'brokenfilesdata'
try:
    os.mkdir(dest)
except Exception as e:
    print(e)

[WinError 183] Cannot create a file when that file already exists: 'brokenfilesdata'


In [9]:
file1 = pq.ParquetFile('dataset/top_gun_opendata_0.parquet')
file1.num_row_groups
file1.metadata
t1 = time.time()
file1.read_row_group(0)
t2  = time.time()
time_to_read = t2-t1

In [11]:
(time_to_read*150327)/60

16.709554874897

In [13]:
perfile = (150327*4*125*125*32)/(8*1024*1024*1024)
perfile

35.00073216855526

In [16]:
perbroken = 150327//20
perbrokendata = (perbroken*4*125*125*32)/(8*1024*1024*1024)
perbrokendata

1.7499551177024841

In [12]:
def split_parquetfiles(path, no_of_files, dest):

        import os
        import math
        import gc
        import pyarrow.parquet as pq
        import pyarrow as pa

        if no_of_files <= 0:
                raise ValueError("no_of_files must be greater than 0")

        os.makedirs(dest, exist_ok=True)

        pf = pq.ParquetFile(path)
        total_rows = pf.metadata.num_rows

        if total_rows == 0:
                return

        rows_per_file = math.ceil(total_rows / no_of_files)

        file_stem = os.path.splitext(os.path.basename(path))[0]

        buffer = []
        buffer_rows = 0
        file_idx = 0

        for batch in pf.iter_batches(batch_size=128):

                buffer.append(batch)
                buffer_rows += batch.num_rows

                if buffer_rows >= rows_per_file:

                        table = pa.Table.from_batches(buffer)

                        out_file = os.path.join(
                                dest,
                                f"{file_stem}_part_{file_idx:05d}.parquet"
                        )

                        pq.write_table(table, out_file)

                        # clear memory
                        del table
                        buffer.clear()
                        buffer_rows = 0
                        file_idx += 1
                        gc.collect()

        if buffer:

                table = pa.Table.from_batches(buffer)

                out_file = os.path.join(
                        dest,
                        f"{file_stem}_part_{file_idx:05d}.parquet"
                )

                pq.write_table(table, out_file)

                del table
                buffer.clear()
                gc.collect()

In [15]:

for file in os.listdir(path):
    if file == 'top_gun_opendata_0.parquet' :
        continue
    split_parquetfiles('dataset/'+file,20,dest)

In [14]:
for file in os.listdir(path):
    print(file)

top_gun_opendata_0.parquet
top_gun_opendata_1.parquet
top_gun_opendata_2.parquet
top_gun_opendata_3.parquet
top_gun_opendata_4.parquet
top_gun_opendata_5.parquet
top_gun_opendata_6.parquet
